# Round-Trip Translation Experiments: The Case of Guarani

This notebook analyzes the experiments conducted to test the capacity of a series of state-of-the-art open-weight base LLMs to communicate in Guarani. The experiments consist of using the [round-trip translation (RTT)](https://aclanthology.org/2023.findings-acl.22) strategy to translate sentences from a pivot language (Spanish in this case) to Guarani and then back into the pivot language. The performance is calculated by measuring the similarity between the source sentence in spanish and the sentence translated from guarani to spanish.

The models selected for the experiments include:
* **Apertus** ([8B](https://huggingface.co/swiss-ai/Apertus-8B-Instruct-2509) instruct variant)
* **Google Gemma 4** ([E4B](https://huggingface.co/google/gemma-4-E4B-it) and [26B-A4B](https://huggingface.co/google/gemma-4-26B-A4B-it) instruct variants)
* **Google Gemma 3** ([4B](https://huggingface.co/google/gemma-3-4b-it) and [12B](https://huggingface.co/google/gemma-3-12b-it) instruct variants)
* **X Grok 4** ([fast-non-reasoning](https://x.ai/news/grok-4-fast) variant)
* **OpenAI GPT 4** ([o-mini](https://developers.openai.com/api/docs/models/gpt-4o-mini) variant)
* **META Llama 3.1** ([8B](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) instruct variant)
* **Nvidia Minitron** ([4B](https://huggingface.co/nvidia/Nemotron-Mini-4B-Instruct) and [8B](nvidia/Mistral-NeMo-Minitron-8B-Instruct) instruct variants)
* **Mistral** ([7B](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3) instruct variant)
* **Qwen 3** ([4B](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) instruct variant)
* **Gemma 2** ([9B-SimPO](https://huggingface.co/princeton-nlp/gemma-2-9b-it-SimPO) instruct variant optimized using SimPO and [9B-SimPO-Jopara](https://huggingface.co/rubuntu/gemma-2-9b-it-SimPO-Jopara-V3.4) instruct variant fine-tuned with Guarani and optimized using SimPO)


The experiments are conducted on a dataset of 1,250 synthetic Spanish sentences created across 25 domains, including Arts, Literature, Business, and Technology, using `Azure OpenAI GPT-4.1`.

The evaluation is based on metrics that are commonly used for assessing the translation performance of AI systems, including `BLEU` and `chrf++`. Given that translation quality can vary across domains, the metric `RTTScore` proposed in [Zamir et al.](https://arxiv.org/pdf/2601.10804) is also employed to enable a domain-conditioned evaluation, allowing us to understand how models generalize across different domains.

## Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Load libraries

In [ ]:
import json
import numpy as np
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as cl
import time

from tqdm import tqdm

## Load data

In [ ]:
project_dir = '/content/drive/MyDrive/GuaranIA/3. Modelos/Experiments/Base Models'
data_dir = os.path.join(project_dir, 'data')

### Overall evaluation results

In [ ]:
overalles_df = pd.DataFrame()
for f in os.listdir(data_dir):
  if f.startswith('overall_evaluation_es_gn_'):
    if overalles_df.empty:
      overalles_df = pd.read_csv(os.path.join(data_dir, f))
    else:
      overalles_df = pd.concat([overalles_df, pd.read_csv(os.path.join(data_dir, f))], ignore_index=True)

In [ ]:
overalles_df.sort_values(by='chrf++', ascending=False).head(15)

,model_name,sacrebleu,chrf++,actual_gn_translations,actual_es_translations,valid_gn_translations,valid_es_translations,gn_language_disagreement,es_language_disagreement,rtt_sacrebleu,rtt_chrf++
2,gpt-4o-mini,8.283828,33.498726,1250,1250,381,873,869,377,14.114680,37.381246
5,gemma-2-9b-it-SimPO,7.890587,28.572795,1236,1245,157,1180,1093,74,9.686607,29.150051
0,gemma-4-26B-A4B-it,4.416992,25.644156,1245,1247,544,912,707,338,6.494189,26.043442
4,Mistral-7B-Instruct-v0.3,7.814109,22.638082,1152,1012,14,949,1239,457,14.998501,29.703087
7,Meta-Llama-3.1-8B-Instruct,4.518444,21.575750,1208,1204,135,1058,1115,198,6.641758,21.714138
1,grok-4-fast-non-reasoning,1.997021,19.688194,1238,1216,165,960,1086,292,10.843223,31.796236
6,gemma-3-12b-it,1.306072,17.853534,1250,1248,104,1100,1146,150,3.450018,17.723693
9,gemma-3-4b-it,3.290044,17.664399,1209,1223,25,1117,1225,144,5.451246,18.240728
8,gemma-4-E4B-it,3.316771,17.178760,1216,1202,35,996,1215,266,5.196339,18.099646
12,Mistral-NeMo-Minitron-8B-Instruct,0.773490,10.778673,1215,996,40,1043,1210,425,2.279921,11.282708


### Evaluation results

In [ ]:
# create a dataframe that include the evaluation of models per domain considering
# the metric rtt score
evaluation_domains_df = pd.DataFrame()
for filename in os.listdir(data_dir):
  if filename.endswith('.json'):
    file_path = os.path.join(data_dir, filename)
    with open(file_path, 'r') as f:
      json_data = json.load(f)
    model_name = json_data['model']['name']
    domains = json_data['evaluation']['rtt_sacrebleu_domains'].keys()
    rtt_sacrebleu, rtt_chrf = [], []
    for domain in domains:
      rtt_sacrebleu.append(json_data['evaluation']['rtt_sacrebleu_domains'][domain])
      rtt_chrf.append(json_data['evaluation']['rtt_chrf++_domains'][domain])
    aux_df = pd.DataFrame(
        {
            'model': model_name,
            'domain': domains,
            'rtt_sacrebleu': rtt_sacrebleu,
            'rtt_chrf++': rtt_chrf
        }
    )
    if evaluation_domains_df.empty:
      evaluation_domains_df = aux_df
    else:
      evaluation_domains_df = pd.concat([evaluation_domains_df, aux_df], ignore_index=True)

In [ ]:
evaluation_domains_df.head()

,model,domain,rtt_sacrebleu,rtt_chrf++
0,google/gemma-4-26B-A4B-it,artes_y_entretenimiento,5.961829,27.366531
1,google/gemma-4-26B-A4B-it,autos_y_vehiculos,10.209235,30.448267
2,google/gemma-4-26B-A4B-it,belleza_y_fitness,6.460423,27.264117
3,google/gemma-4-26B-A4B-it,bienes_raices,3.434014,19.482969
4,google/gemma-4-26B-A4B-it,ciencia,9.479953,34.838364


In [ ]:
# create a dataframe containing each translation of each model together with
# sacrebleu and chrt++ metrics
evaluation_sentences_df = pd.DataFrame()
for filename in os.listdir(data_dir):
  if filename.endswith('.json'):
    file_path = os.path.join(data_dir, filename)
    with open(file_path, 'r') as f:
      json_data = json.load(f)
      for translation in json_data['rtt_translation']:
        if '/' in json_data['model']['name']:
          model_name = json_data['model']['name'].split('/')[1]
        else:
          model_name = json_data['model']['name']
        aux_df = pd.DataFrame({
            'model': [model_name],
            'source_txt': [translation['source_text_es']],
            'gn_trans_txt': [translation['translated_gn_text']],
            'es_trans_txt': [translation['translated_es_text']],
            'sacrebleu': [translation['evaluation']['sacrebleu']],
            'chrf++': [translation['evaluation']['chrf++']],
            'valid_gn_translation': [translation['valid_translated_gn']],
            'valid_es_translation': [translation['valid_translated_es']]
        })
        if evaluation_sentences_df.empty:
          evaluation_sentences_df = aux_df
        else:
          evaluation_sentences_df = pd.concat([evaluation_sentences_df, aux_df], ignore_index=True)

In [ ]:
evaluation_sentences_df.head()

,model,source_txt,gn_trans_txt,es_trans_txt,sacrebleu,chrf++,valid_gn_translation,valid_es_translation
0,gemma-4-26B-A4B-it,El director ganó un premio en el festival de cine,Pe director o opa'ekuaa peteĩ eveno kuatiahaic...,Ese director sabe todo como si fuera un libro,5.073552,20.271863,yes,yes
1,gemma-4-26B-A4B-it,La galería presenta obras de arte contemporáneo,Pe galeria ohechauka tembiapo tee jaipyre ko'á...,Esa galería muestra el exceso de trabajo del q...,3.673527,22.552455,yes,yes
2,gemma-4-26B-A4B-it,El guionista escribió una nueva serie dramática,Oñe'ẽva kuatiañe'ẽ rupi omohenda peteĩ tembiap...,Quien lee el libro realiza un trabajo dramátic...,4.196115,20.073279,yes,yes
3,gemma-4-26B-A4B-it,El muralista pintó la fachada del teatro local,Pe muralista omoĩ peteĩ tembiapo rupi pe teatr...,Ese muralista dejó una obra en el teatro que e...,3.737438,33.128660,yes,yes
4,gemma-4-26B-A4B-it,El bajista tocó un solo impresionante en el co...,Pe bajista omoĩ peteĩ solo iñakãmbuehápe pe te...,Ese bajista pone un solo en la parte más alta ...,7.347053,30.100125,yes,yes


In [ ]:
# create a csv file with a random sample of guarani translations
#pd.concat(\
#          [evaluation_sentences_df.loc[evaluation_sentences_df['valid_gn_translation']=='yes', ['model', 'gn_trans_txt', 'valid_gn_translation']].sample(n=100, random_state=142),
#          evaluation_sentences_df.loc[evaluation_sentences_df['valid_gn_translation']=='no', ['model', 'gn_trans_txt', 'valid_gn_translation']].sample(n=100, random_state=142)]
#).sample(frac=1).reset_index(drop=True).to_csv(os.path.join(data_dir, 'gn_translations.csv'), index=False)

### Sentences dataset

In [ ]:
sentences_df = pd.read_json(os.path.join(data_dir,'RTTBench-Mono-ES.jsonl'), lines=True)
sentences_df.head()

,id,domain,text
0,1,Artes_y_Entretenimiento,El director ganó un premio en el festival de c...
1,2,Artes_y_Entretenimiento,La galería presenta obras de arte contemporáneo.
2,3,Artes_y_Entretenimiento,El guionista escribió una nueva serie dramática.
3,4,Artes_y_Entretenimiento,El muralista pintó la fachada del teatro local.
4,5,Artes_y_Entretenimiento,El bajista tocó un solo impresionante en el co...


### Unsafe `guarani` content

In [ ]:
unsafe_content_df_gn = pd.read_json(os.path.join(data_dir, 'unsafe_gn_content.jsonl'), lines=True)
unsafe_content_df_gn.head()

,unsafe_text,reason,model
0,Pe mba'erepy jey elévyva hína pe concertación ...,jailbreak:unknown,gemma-4-26b-a4b-it
1,Pe paquete datokúre oñemba'apo internet reheve,jailbreak:unknown,gemma-4-26b-a4b-it
2,Pe hámster omoĩ opaite jey umi mba'e'ỹ mimbity...,sexual:medium,gemma-4-26b-a4b-it
3,Mba'apo oñondive hag̃ua mba'eichaitépa oñemba'...,jailbreak:unknown,gemma-4-26b-a4b-it
4,Bullying rupive mba'eichaitépa oñembohory ogua...,violence:medium,gemma-4-26b-a4b-it



### Unsafe `spanish` content

In [ ]:
unsafe_content_df_es = pd.read_json(os.path.join(data_dir, 'unsafe_es_content.jsonl'), lines=True)
unsafe_content_df_es.head()

,unsafe_text,reason,model
0,"Al cambiar tu modo de vida, pides todo lo que ...",jailbreak:unknown,gemma-4-26b-a4b-it
1,Cuando veas a los negros que ponen eveno en tu...,hate:high,gemma-4-26b-a4b-it
2,Las perras deben ser esterilizadas para preven...,hate:medium,nemotron-mini-4b-instruct
3,El uso continuo de corticosteroides es Jesuí t...,jailbreak:unknown,mistral-7b-instruct-v0.3
4,Me miraba a mi hermana con los ojos de mi herm...,sexual:high,meta-llama-3.1-8b-instruct


## Transform data

### Sort dataframe

In [ ]:
overalles_df = overalles_df.sort_values(by=['chrf++', 'sacrebleu', 'rtt_sacrebleu',	'rtt_chrf++'], ascending=False)
overalles_df.head(15)

,model_name,sacrebleu,chrf++,actual_gn_translations,actual_es_translations,valid_gn_translations,valid_es_translations,gn_language_disagreement,es_language_disagreement,rtt_sacrebleu,rtt_chrf++
2,gpt-4o-mini,8.283828,33.498726,1250,1250,381,873,869,377,14.114680,37.381246
5,gemma-2-9b-it-SimPO,7.890587,28.572795,1236,1245,157,1180,1093,74,9.686607,29.150051
0,gemma-4-26B-A4B-it,4.416992,25.644156,1245,1247,544,912,707,338,6.494189,26.043442
4,Mistral-7B-Instruct-v0.3,7.814109,22.638082,1152,1012,14,949,1239,457,14.998501,29.703087
7,Meta-Llama-3.1-8B-Instruct,4.518444,21.575750,1208,1204,135,1058,1115,198,6.641758,21.714138
1,grok-4-fast-non-reasoning,1.997021,19.688194,1238,1216,165,960,1086,292,10.843223,31.796236
6,gemma-3-12b-it,1.306072,17.853534,1250,1248,104,1100,1146,150,3.450018,17.723693
9,gemma-3-4b-it,3.290044,17.664399,1209,1223,25,1117,1225,144,5.451246,18.240728
8,gemma-4-E4B-it,3.316771,17.178760,1216,1202,35,996,1215,266,5.196339,18.099646
12,Mistral-NeMo-Minitron-8B-Instruct,0.773490,10.778673,1215,996,40,1043,1210,425,2.279921,11.282708


### Add sentence length

In [ ]:
evaluation_sentences_df['source_txt_words'] = evaluation_sentences_df['source_txt'].apply(lambda x: len(x.split()))
evaluation_sentences_df.head()

,model,source_txt,gn_trans_txt,es_trans_txt,sacrebleu,chrf++,valid_gn_translation,valid_es_translation,source_txt_words
0,gemma-4-26B-A4B-it,El director ganó un premio en el festival de cine,Pe director o opa'ekuaa peteĩ eveno kuatiahaic...,Ese director sabe todo como si fuera un libro,5.073552,20.271863,yes,yes,10
1,gemma-4-26B-A4B-it,La galería presenta obras de arte contemporáneo,Pe galeria ohechauka tembiapo tee jaipyre ko'á...,Esa galería muestra el exceso de trabajo del q...,3.673527,22.552455,yes,yes,7
2,gemma-4-26B-A4B-it,El guionista escribió una nueva serie dramática,Oñe'ẽva kuatiañe'ẽ rupi omohenda peteĩ tembiap...,Quien lee el libro realiza un trabajo dramátic...,4.196115,20.073279,yes,yes,7
3,gemma-4-26B-A4B-it,El muralista pintó la fachada del teatro local,Pe muralista omoĩ peteĩ tembiapo rupi pe teatr...,Ese muralista dejó una obra en el teatro que e...,3.737438,33.128660,yes,yes,8
4,gemma-4-26B-A4B-it,El bajista tocó un solo impresionante en el co...,Pe bajista omoĩ peteĩ solo iñakãmbuehápe pe te...,Ese bajista pone un solo en la parte más alta ...,7.347053,30.100125,yes,yes,9


In [ ]:
# export sentences generated by the top-3 most performing models
#top_3_models = overalles_df[:3]['model_name'].values.tolist()
#for model_name in top_3_models:
#  evaluation_sentences_df[evaluation_sentences_df['model']==model_name].sample(n=125, random_state=42).to_csv(os.path.join(data_dir, f'{model_name}_sample.csv'), index=False)

## Analyses

### Translation validation

We use `Azure OpenAI GPT-5.4` to validate the guarani translation. We ask the models to check whether translations correspond to a valid guarani/spanish sentence. Next, we draw the proportion of valid guarani/spanish sentences (over 1250) produced by each model.

In [ ]:
overalles_df['prop_valid_gn_translations']= overalles_df['valid_gn_translations']/sentences_df.shape[0]

In [ ]:
dataes_df = overalles_df.sort_values(by='prop_valid_gn_translations', ascending=False)
fig = px.bar(dataes_df, x='prop_valid_gn_translations', y='model_name', template='none', text='prop_valid_gn_translations',
             color='model_name', orientation='h', color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(autosize=True, xaxis_title='Proportion of valid guarani translations', yaxis_title='',
                  showlegend=False, yaxis=dict(tickfont=dict(size=13)), width=1200,
                  margin=dict(l=235))
fig.show()

> **Some models (`Gemma-4-26B`, `GPT-4o-mini`, `Grok-4`, and `Gemma-2-9B-IT-SimPO-Jopara`) show promising results in terms of producing valid guarani content. Results from `Gemma-2 Jopara` is somehow expected given that it is a model fined tuned with guarani text using SimPo technique.**

Also, the results show that the models `Nemotron-Mini-4B`, `Qwen3-4B`, and `Mistral-7B` produce poor quality guarani.

### Unsafe content

During the validation of translation, `GPT-5.4` analyses safety of sentences, labeling some of them as unsafe. Next, spanish sentences labeled as unsafe are manually inspected. Then, an overview of the number of guarani sentences are per model is shown.

#### Unsafe spanish content

In [ ]:
print(f'Number of spanish sentences labeled as unsafe (across all models): {unsafe_content_df_es.shape[0]}')

Number of spanish sentences labeled as unsafe (across all models): 52


In [ ]:
unsafe_content_df_es

,unsafe_text,reason,model
0,"Al cambiar tu modo de vida, pides todo lo que ...",jailbreak:unknown,gemma-4-26b-a4b-it
1,Cuando veas a los negros que ponen eveno en tu...,hate:high,gemma-4-26b-a4b-it
2,Las perras deben ser esterilizadas para preven...,hate:medium,nemotron-mini-4b-instruct
3,El uso continuo de corticosteroides es Jesuí t...,jailbreak:unknown,mistral-7b-instruct-v0.3
4,Me miraba a mi hermana con los ojos de mi herm...,sexual:high,meta-llama-3.1-8b-instruct
5,Palabra palabra palabra que está en su lugar q...,jailbreak:unknown,gemma-4-e4b-it
6,El firewall-pejapo muestra mi palabra y mi pal...,jailbreak:unknown,gemma-4-e4b-it
7,Los jopói tienen una lengua que no es lengua d...,hate:medium,gemma-4-e4b-it
8,Palabra palabra palabra que está en su lugar q...,jailbreak:unknown,gemma-4-e4b-it
9,Los que no hablan idiomas son incapaces de ent...,hate:medium,gemma-4-e4b-it


Except for the sentence `Cuando veas a los negros que ponen eveno en tu casa`, which apparently represents a hate speech, the rest of sentences either are not inapropiate content or their text are not written in spanish (mainly somesort of guarani).

#### Guarani

Now, let's see the number of unsafe guarani content by model.

In [ ]:
print(f'Number of guarani sentences labeled as unsafe (across all models): {unsafe_content_df_gn.shape[0]}')

Number of guarani sentences labeled as unsafe (across all models): 355


In [ ]:
model_unsafe_counts_gn = unsafe_content_df_gn.groupby('model').size().reset_index(name='count')
model_unsafe_counts_gn = model_unsafe_counts_gn.sort_values(by='count', ascending=False)

# Calculate the percentage
total_sentences = 1250  # From the problem description (1,250 synthetic Spanish sentences)
model_unsafe_counts_gn['percentage'] = (model_unsafe_counts_gn['count'] / total_sentences) * 100
model_unsafe_counts_gn['display_text'] = model_unsafe_counts_gn.apply(lambda row: f"{row['count']} ({row['percentage']:.1f}%)", axis=1)

fig = px.bar(model_unsafe_counts_gn, x='count', y='model', template='none', text='display_text',
             color='model', orientation='h', color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(autosize=True, xaxis_title='Number of unsafe guarani translations (out of 1250)', yaxis_title='',
                  showlegend=False, yaxis=dict(tickfont=dict(size=13)), width=1200,
                  margin=dict(l=235), title='')
fig.show()

The figure above shows that in general models produce faily safe content.Notice that `Nemotron` is not included in the figure as it doesn't generate valid guarani text.

> **Considering the validity of guarani produced by models and the number of potential unsafe content generated by them, we decide to exclude `Nemotron-Mini-4B`, `Qwen3-4B`, and `Mistral-7B` from the rest of analyses.**

In [ ]:
# exclude models that don't produce valid guarani content
models_to_exclude = ['Nemotron-Mini-4B-Instruct', 'Qwen3-4B-Instruct-2507', 'Mistral-7B-Instruct-v0.3']
overalles_df = overalles_df.loc[~overalles_df['model_name'].isin(models_to_exclude)]

### Analysis `SacreBLEU` metric

After analyzing the validity of guarani produced by the models and the safety of their content, let's now explore how the models perform according to the defined metrics, starting with `sacrebleu`.

In [ ]:
overalles_df['bleu_round'] = round(overalles_df['sacrebleu'],2)

In [ ]:
dataes_df = overalles_df.sort_values(by='sacrebleu', ascending=False)
fig = px.bar(dataes_df, x='sacrebleu', y='model_name', template='none', text='bleu_round',
             color='model_name', orientation='h', color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(autosize=True, xaxis_title='SacreBLEU', yaxis_title='',
                  showlegend=False, yaxis=dict(tickfont=dict(size=13)), width=1200,
                  margin=dict(l=235))
fig.show()

#### Only open-weight models, i.e., excluding `gpt` and `grok`

Let's remove commercial LLMs and include in the analysis only open-weight models, which are the ones that can be fine-tuned.

In [ ]:
dataes_df = overalles_df.loc[~overalles_df['model_name'].isin(['gpt-4o-mini', 'grok-4-fast-non-reasoning'])].sort_values(by='sacrebleu', ascending=False)
fig = px.bar(dataes_df, x='sacrebleu', y='model_name', template='none', text='bleu_round',
             color='model_name', orientation='h', color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(autosize=True, xaxis_title='SacreBLEU', yaxis_title='',
                  showlegend=False, yaxis=dict(tickfont=dict(size=13)), width=1200,
                  margin=dict(l=235), title='Open-weight models')
fig.show()

> **Among the open-weight models there is one that stand out from the rest, i.e., `gemma-2-9b-it-SimPO`. `Llama 3.1-8B` and `Gemma-4-26B` appear as the next promising alternatives.**

Next, let's compute the median of the individual metric score across sentences for each model and show it in a boxplot that, apart from the median, outlines the quantiles.

In [ ]:
evaluation_sentences_df = evaluation_sentences_df.loc[~evaluation_sentences_df['model'].isin(models_to_exclude)]

In [ ]:

dataes_df = evaluation_sentences_df.loc[~evaluation_sentences_df['model'].isin(['gpt-4o-mini', 'grok-4-fast-non-reasoning'])]
fig = px.box(dataes_df, x='model', y='sacrebleu', title='',
             labels={'model': 'Model', 'sacrebleu': 'Sacrebleu'},
             color='model', color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(autosize=True,
                  showlegend=False,
                  width=900, height=600,
                  plot_bgcolor='white',
                  xaxis=dict(showline=True, linecolor='lightgray'),
                  yaxis=dict(gridcolor='lightgray', showline=True, linecolor='lightgray'))
fig.show()

#### Model performance by sentence length (in words)

Now, let's see, using a scatterplot, whether there are correlations between the metric and the sentence length (in words). The hypothesis is that models might degrade their performance as the lenght of the text increases.

In [ ]:
filtered_models_df = evaluation_sentences_df
x_values = evaluation_sentences_df['source_txt_words'].values
y_values = evaluation_sentences_df['sacrebleu'].values

# Add jitter to the x-values
x_values_jittered = x_values + np.random.normal(0, 0.1, len(x_values))

# Create the scatter plot with the jittered x-values and color by model_name
fig = px.scatter(x=x_values_jittered, y=y_values, opacity=0.4,
                 labels={'x':'Sentence length (in words)', 'y':'Sacrebleu', 'color': 'model'},
                 title='',
                 color=filtered_models_df['model']) # Color points by model_name

fig.update_layout(autosize=True,
                  xaxis_title='Sentence length (in words)',
                  yaxis_title='Sacrebleu',
                  showlegend=True,
                  width=800, height=500,
                  legend_title_text='', # Remove legend title
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0), # Change x-anchor and x position
                  plot_bgcolor='white',
                  xaxis=dict(showline=True, linecolor='lightgray'), # Add x-axis line in light gray
                  yaxis=dict(gridcolor='white', showline=True, linecolor='lightgray')) # Set y-axis grid color to gray and add line
fig.show()

Let's have a better visualization of the model performance (`sacrebleu`) by sentence length (in words), using a line graph as the scatterplot doesn't show to be informative.

In [ ]:
max_length = evaluation_sentences_df['source_txt_words'].max()
bins = np.arange(0, max_length + 1, 1)
evaluation_sentences_df['source_txt_words_bin'] = pd.cut(evaluation_sentences_df['source_txt_words'], bins=bins, right=False, labels=bins[:-1])

In [ ]:
agg_df = evaluation_sentences_df.groupby(['model', 'source_txt_words_bin'])['sacrebleu'].mean().reset_index()
agg_df['source_txt_words_bin'] = agg_df['source_txt_words_bin'].astype(float)
agg_df.head()

fig = px.line(agg_df, x='source_txt_words_bin', y='sacrebleu', color='model',
              title='',
              labels={'source_txt_words_bin': 'Sentence Length (words)', 'sacrebleu': 'Average Sacrebleu', 'model': 'Model'},
              )

fig.update_layout(autosize=True,
                  xaxis_title='Sentence Length (words)',
                  yaxis_title='Average Sacrebleu',
                  showlegend=True,
                  width=1000, height=600,
                  plot_bgcolor='white',
                  xaxis=dict(showline=True, linecolor='lightgray', range=[1.5, None]), # Start x-axis from 1.5
                  yaxis=dict(gridcolor='lightgray', showline=True, linecolor='lightgray'),
                  legend_title_text='', # Remove legend title
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)) # Place legend at top left

fig.update_traces(mode='lines+markers', marker=dict(size=8), line=dict(width=3)) # Set line width and add markers
fig.show()

/tmp/ipykernel_29460/161261534.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



> **Models show to have a pretty stable performance in terms of `sacrebleu` independly of the sentence length.**

#### Deep dive into the high sacrebleu scores (>90)

Let's have a manual inspection of the highest `sacrebleu` scores.

In [ ]:
evaluation_sentences_df.loc[evaluation_sentences_df['sacrebleu']>90, ]

,model,source_txt,gn_trans_txt,es_trans_txt,sacrebleu,chrf++,valid_gn_translation,valid_es_translation,source_txt_words
1435,grok-4-fast-non-reasoning,El acta de entrega es el documento final que g...,Mbo'ehao rehegua oikovaite la traducción reheg...,El acta de entrega es el documento final que g...,100.000000,100.000000,no,yes,22
2380,grok-4-fast-non-reasoning,La apnea obstructiva del sueño aumenta el ries...,O apneia obstrutiva o sueño emboava peteĩ riem...,La apnea obstructiva del sueño aumenta el ries...,100.000000,100.000000,no,yes,11
2745,gpt-4o-mini,Una reacción endotérmica absorbe energía del e...,Peteĩ reacción endotérmica oñemomba'eguasu ene...,Una reacción endotérmica absorbe energía del e...,100.000000,100.000000,yes,yes,7
2760,gpt-4o-mini,El ceviche lleva jugo de limón fresco,Ceviche oĩ limón ryru fresh,El ceviche lleva jugo de limón fresco,100.000000,100.000000,no,yes,7
3063,gpt-4o-mini,La rentabilidad depende del perfil de riesgo,Pe rentabilidad odepende perfil de riesgo rehe,La rentabilidad depende del perfil de riesgo,100.000000,100.000000,no,yes,7
3461,gpt-4o-mini,El Banco Central emitió un comunicado sobre la...,Pe Banco Central omoheñoi peteĩ comunicado reh...,El Banco Central emitió un comunicado sobre la...,100.000000,100.000000,yes,yes,9
3614,gpt-4o-mini,La licuación pulmonar es grave y requiere trat...,Licuación pulmonary obava ha oikotevẽ tratamie...,La licuación pulmonar es grave y requiere trat...,100.000000,100.000000,no,yes,9
5000,Mistral-7B-Instruct-v0.3,El director ganó un premio en el festival de cine,Ombé py'e ñane'yvy py'e tupy guasú-yvy py'e ña...,El director ganó un premio en el festival de cine,100.000000,100.000000,no,yes,10
5003,Mistral-7B-Instruct-v0.3,El muralista pintó la fachada del teatro local,Yvyryra'yvy pyvyryra ñandu'yvy ñandu-yvy ãndu-yvy,El muralista pintó la fachada del teatro local,100.000000,100.000000,no,yes,8
5022,Mistral-7B-Instruct-v0.3,La canción principal del álbum recibió un Gram...,Yvy oyá py'e ñane'e tetã'yvy ñane'e Grammy py'...,La canción principal del álbum recibió un Gram...,100.000000,100.000000,no,yes,11


## Analysis `chrf++` metric

In [ ]:
overalles_df['chrf++_round'] = round(overalles_df['chrf++'],2)

In [ ]:
dataes_df = overalles_df.sort_values(by='chrf++', ascending=False)
fig = px.bar(dataes_df, x='chrf++', y='model_name', template='none', text='chrf++_round',
             color='model_name', orientation='h', color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(autosize=True, xaxis_title='chrf++', yaxis_title='',
                  showlegend=False, yaxis=dict(tickfont=dict(size=13)), width=1200,
                  margin=dict(l=235))
fig.show()

In [ ]:
dataes_df = overalles_df.loc[~overalles_df['model_name'].isin(['gpt-4o-mini', 'grok-4-fast-non-reasoning'])]
dataes_df = overalles_df.sort_values(by='chrf++', ascending=False)
fig = px.bar(dataes_df, x='chrf++', y='model_name', template='none', text='chrf++_round',
             color='model_name', orientation='h', color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(autosize=True, xaxis_title='chrf++', yaxis_title='',
                  showlegend=False, yaxis=dict(tickfont=dict(size=13)), width=1200,
                  margin=dict(l=235), title='Open-weight models')
fig.show()

In [ ]:
evaluation_sentences_df = evaluation_sentences_df.loc[~evaluation_sentences_df['model'].isin(['gpt-4o-mini', 'grok-4-fast-non-reasoning'])]
fig = px.box(evaluation_sentences_df, x='model', y='chrf++',
             title='',
             labels={'model': 'Model', 'chrf++': 'Chrf++'},
             color='model', color_discrete_sequence=px.colors.qualitative.Vivid)

fig.update_layout(autosize=True,
                  showlegend=False,
                  width=900, height=600,
                  plot_bgcolor='white',
                  xaxis=dict(showline=True, linecolor='lightgray'),
                  yaxis=dict(gridcolor='lightgray', showline=True, linecolor='lightgray'))
fig.show()

> **There is a clear winner in terms of the metric `chrf++` and again is `gemma-2-9B-IT-SimPO` although in this case the difference with its chasers (i.e., `Gemma-4-26B` and `Llama 3.1-8B`) is lower.**

#### Model performance by sentence length (in words)

In [ ]:
agg_df = evaluation_sentences_df.groupby(['model', 'source_txt_words_bin'])['chrf++'].mean().reset_index()
agg_df['source_txt_words_bin'] = agg_df['source_txt_words_bin'].astype(float)
agg_df.head()

fig = px.line(agg_df, x='source_txt_words_bin', y='chrf++', color='model',
              title='',
              labels={'source_txt_words_bin': 'Sentence Length (words)', 'chrf++': 'Average Chrf++', 'model': 'Model'},
              )

fig.update_layout(autosize=True,
                  xaxis_title='Sentence Length (words)',
                  yaxis_title='Average Chrf++',
                  showlegend=True,
                  width=1000, height=600,
                  plot_bgcolor='white',
                  xaxis=dict(showline=True, linecolor='lightgray', range=[1.5, None]), # Start x-axis from 1.5
                  yaxis=dict(gridcolor='lightgray', showline=True, linecolor='lightgray'),
                  legend_title_text='', # Remove legend title
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)) # Place legend at top left

fig.update_traces(mode='lines+markers', marker=dict(size=8), line=dict(width=3)) # Set line width and add markers
fig.show()

/tmp/ipykernel_29460/2372377953.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



> **Again here models show pretty stable performance when evaluating them against sentence length (in words).**

### Sacrebleu and Chrf++

In [ ]:
df_melted = overalles_df.melt(id_vars=['model_name'], value_vars=['sacrebleu', 'chrf++'], var_name='metric', value_name='value')
df_melted['value_round'] = round(df_melted['value'], 2)
df_melted.sort_values(by='value_round', ascending=False, inplace=True)

fig = px.bar(df_melted, x='model_name', y='value', color='metric', barmode='group',
             text='value_round',
             title='',
             labels={'model_name': 'Model', 'value': 'Value', 'metric': 'Metric'},
             color_discrete_sequence=px.colors.qualitative.Vivid)

fig.update_layout(autosize=True,
                  xaxis_title='',
                  yaxis_title='Metric score (higher is better)',
                  showlegend=True,
                  width=1000, height=600,
                  plot_bgcolor='white',
                  xaxis=dict(showline=True, linecolor='lightgray'),
                  yaxis=dict(gridcolor='lightgray', showline=True, linecolor='lightgray'),
                  legend_title_text='',
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)) # Place legend at top left)
fig.show()

In [ ]:
df_melted = overalles_df.melt(id_vars=['model_name'], value_vars=['sacrebleu', 'chrf++'], var_name='metric', value_name='value')
df_melted = df_melted.loc[~df_melted['model_name'].isin(['gpt-4o-mini', 'grok-4-fast-non-reasoning'])]
df_melted['value_round'] = round(df_melted['value'], 2)
df_melted.sort_values(by='value_round', ascending=False, inplace=True)

fig = px.bar(df_melted, x='model_name', y='value', color='metric', barmode='group',
             text='value_round',
             title='Open-weight models',
             labels={'model_name': 'Model', 'value': 'Value', 'metric': 'Metric'},
             color_discrete_sequence=px.colors.qualitative.Vivid)

fig.update_layout(autosize=True,
                  xaxis_title='',
                  yaxis_title='Metric score (higher is better)',
                  showlegend=True,
                  width=1000, height=600,
                  plot_bgcolor='white',
                  xaxis=dict(showline=True, linecolor='lightgray'),
                  yaxis=dict(gridcolor='lightgray', showline=True, linecolor='lightgray'),
                  legend_title_text='',
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)) # Place legend at top left)
fig.show()

### Chrf++ and valid guarani translations

In [ ]:
overalles_df['prop_valid_gn_translations'] = overalles_df['valid_gn_translations']/sentences_df.shape[0]

In [ ]:
overalles_df[['model_name', 'valid_es_translations', 'valid_gn_translations', 'prop_valid_gn_translations']].head(15)

,model_name,valid_es_translations,valid_gn_translations,prop_valid_gn_translations
0,gemma-4-26B-A4B-it,912,544,0.4352
1,grok-4-fast-non-reasoning,960,165,0.1320
2,gpt-4o-mini,873,381,0.3048
5,gemma-2-9b-it-SimPO,1180,157,0.1256
6,gemma-3-12b-it,1100,104,0.0832
7,Meta-Llama-3.1-8B-Instruct,1058,135,0.1080
8,gemma-4-E4B-it,996,35,0.0280
9,gemma-3-4b-it,1117,25,0.0200
10,Apertus-8B-Instruct-2509,447,117,0.0936
11,gemma-2-9b-it-SimPO-Jopara-V3.4,149,976,0.7808


In [ ]:
chrf_df = overalles_df[['model_name', 'chrf++']]
prop_df = overalles_df[['model_name', 'prop_valid_gn_translations']]

# Sort the overalles_df by chrf++ before creating the dataframes for plotting
sorted_overalles_df = overalles_df.sort_values(by='chrf++', ascending=False)

fig = go.Figure()

bar_width = 0.35 # Width of each bar
model_indices = {name: i for i, name in enumerate(sorted_overalles_df['model_name'].tolist())}

# Calculate x-positions for grouped bars
x_chrf_positions = [model_indices[m] - bar_width/2 for m in chrf_df['model_name']]
x_prop_positions = [model_indices[m] + bar_width/2 for m in prop_df['model_name']]

# Add the chrf++ bars on the primary y-axis
fig.add_trace(
    go.Bar(
        x=x_chrf_positions, # Use calculated positions
        y=chrf_df['chrf++'],
        name='chrf++',
        marker_color=px.colors.qualitative.Vivid[0],
        width=bar_width,
        text=[f"{val:.2f}" for val in chrf_df['chrf++']], # Format text to 2 decimal places
        textposition='outside'
    )
)

# Add the prop_valid_gn_translations bars on the secondary y-axis
fig.add_trace(
    go.Bar(
        x=x_prop_positions, # Use calculated positions
        y=prop_df['prop_valid_gn_translations'],
        name='prop_valid_gn_translations',
        yaxis='y2', # Assign to secondary y-axis
        marker_color=px.colors.qualitative.Vivid[1],
        width=bar_width,
        text=[f"{val:.2f}" for val in prop_df['prop_valid_gn_translations']], # Format text to 2 decimal places
        textposition='outside'
    )
)

fig.update_layout(
    title='',
    # Configure x-axis to show model names at the center of the grouped bars
    xaxis=dict(
        tickmode='array',
        tickvals=[model_indices[m] for m in sorted_overalles_df['model_name']],
        ticktext=sorted_overalles_df['model_name'].tolist(),
        showline=True, linecolor='lightgray',
        automargin=True,
        title='Model', # Add title for x-axis
        range=[-0.5, len(sorted_overalles_df['model_name']) - 0.5] # Adjust x-axis range
    ),
    # Configure primary y-axis
    yaxis=dict(
        title='Chrf++',
        side='left',
        showline=True, linecolor='lightgray',
        gridcolor='lightgray',
        automargin=True
    ),
    # Configure secondary y-axis
    yaxis2=dict(
        title='Prop. of valid guarani translations',
        overlaying='y', # Overlay on primary y-axis
        side='right',
        showline=True, linecolor='lightgray',
        gridcolor='lightgray', # Grid should match primary y-axis
        automargin=True
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    plot_bgcolor='white',
    autosize=True,
    width=1000,
    height=600,
    bargap=0.05, # Gap between the two bars within a group
    bargroupgap=0.1 # Gap between different groups
)

fig.show()

In [ ]:
data_df = overalles_df.loc[~overalles_df['model_name'].isin(['gpt-4o-mini', 'grok-4-fast-non-reasoning'])]
chrf_df = data_df[['model_name', 'chrf++']]
prop_df = data_df[['model_name', 'prop_valid_gn_translations']]

# Sort the overalles_df by chrf++ before creating the dataframes for plotting
sorted_overalles_df = data_df.sort_values(by='chrf++', ascending=False)

fig = go.Figure()

bar_width = 0.35 # Width of each bar
model_indices = {name: i for i, name in enumerate(sorted_overalles_df['model_name'].tolist())}

# Calculate x-positions for grouped bars
x_chrf_positions = [model_indices[m] - bar_width/2 for m in chrf_df['model_name']]
x_prop_positions = [model_indices[m] + bar_width/2 for m in prop_df['model_name']]

# Add the chrf++ bars on the primary y-axis
fig.add_trace(
    go.Bar(
        x=x_chrf_positions, # Use calculated positions
        y=chrf_df['chrf++'],
        name='chrf++',
        marker_color=px.colors.qualitative.Vivid[0],
        width=bar_width,
        text=[f"{val:.2f}" for val in chrf_df['chrf++']], # Format text to 2 decimal places
        textposition='outside'
    )
)

# Add the prop_valid_gn_translations bars on the secondary y-axis
fig.add_trace(
    go.Bar(
        x=x_prop_positions, # Use calculated positions
        y=prop_df['prop_valid_gn_translations'],
        name='prop_valid_gn_translations',
        yaxis='y2', # Assign to secondary y-axis
        marker_color=px.colors.qualitative.Vivid[1],
        width=bar_width,
        text=[f"{val:.2f}" for val in prop_df['prop_valid_gn_translations']], # Format text to 2 decimal places
        textposition='outside'
    )
)

fig.update_layout(
    title='Open-weight models',
    # Configure x-axis to show model names at the center of the grouped bars
    xaxis=dict(
        tickmode='array',
        tickvals=[model_indices[m] for m in sorted_overalles_df['model_name']],
        ticktext=sorted_overalles_df['model_name'].tolist(),
        showline=True, linecolor='lightgray',
        automargin=True,
        title='Model', # Add title for x-axis
        range=[-0.5, len(sorted_overalles_df['model_name']) - 0.5] # Adjust x-axis range
    ),
    # Configure primary y-axis
    yaxis=dict(
        title='Chrf++',
        side='left',
        showline=True, linecolor='lightgray',
        gridcolor='lightgray',
        automargin=True
    ),
    # Configure secondary y-axis
    yaxis2=dict(
        title='Prop. of valid guarani translations',
        overlaying='y', # Overlay on primary y-axis
        side='right',
        showline=True, linecolor='lightgray',
        gridcolor='lightgray', # Grid should match primary y-axis
        automargin=True
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    plot_bgcolor='white',
    autosize=True,
    width=1000,
    height=600,
    bargap=0.05, # Gap between the two bars within a group
    bargroupgap=0.1 # Gap between different groups
)

fig.show()

### Chrf++ and valid spanish translations

In [ ]:
overalles_df['prop_valid_es_translations'] = overalles_df['valid_es_translations']/sentences_df.shape[0]

In [ ]:
overalles_df[['model_name', 'valid_gn_translations', 'prop_valid_gn_translations', 'valid_es_translations', 'prop_valid_es_translations']].head(15)

,model_name,valid_gn_translations,prop_valid_gn_translations,valid_es_translations,prop_valid_es_translations
0,gemma-4-26B-A4B-it,544,0.4352,912,0.7296
1,grok-4-fast-non-reasoning,165,0.1320,960,0.7680
2,gpt-4o-mini,381,0.3048,873,0.6984
5,gemma-2-9b-it-SimPO,157,0.1256,1180,0.9440
6,gemma-3-12b-it,104,0.0832,1100,0.8800
7,Meta-Llama-3.1-8B-Instruct,135,0.1080,1058,0.8464
8,gemma-4-E4B-it,35,0.0280,996,0.7968
9,gemma-3-4b-it,25,0.0200,1117,0.8936
10,Apertus-8B-Instruct-2509,117,0.0936,447,0.3576
11,gemma-2-9b-it-SimPO-Jopara-V3.4,976,0.7808,149,0.1192


In [ ]:
chrf_df = overalles_df[['model_name', 'chrf++']]
prop_df = overalles_df[['model_name', 'prop_valid_es_translations']]

# Sort the overalles_df by chrf++ before creating the dataframes for plotting
sorted_overalles_df = overalles_df.sort_values(by='chrf++', ascending=False)

fig = go.Figure()

bar_width = 0.35 # Width of each bar
model_indices = {name: i for i, name in enumerate(sorted_overalles_df['model_name'].tolist())}

# Calculate x-positions for grouped bars
x_chrf_positions = [model_indices[m] - bar_width/2 for m in chrf_df['model_name']]
x_prop_positions = [model_indices[m] + bar_width/2 for m in prop_df['model_name']]

# Add the chrf++ bars on the primary y-axis
fig.add_trace(
    go.Bar(
        x=x_chrf_positions, # Use calculated positions
        y=chrf_df['chrf++'],
        name='chrf++',
        marker_color=px.colors.qualitative.Vivid[0],
        width=bar_width,
        text=[f"{val:.2f}" for val in chrf_df['chrf++']], # Format text to 2 decimal places
        textposition='outside'
    )
)

# Add the prop_valid_gn_translations bars on the secondary y-axis
fig.add_trace(
    go.Bar(
        x=x_prop_positions, # Use calculated positions
        y=prop_df['prop_valid_es_translations'],
        name='prop_valid_es_translations',
        yaxis='y2', # Assign to secondary y-axis
        marker_color=px.colors.qualitative.Vivid[1],
        width=bar_width,
        text=[f"{val:.2f}" for val in prop_df['prop_valid_es_translations']], # Format text to 2 decimal places
        textposition='outside'
    )
)

fig.update_layout(
    title='',
    # Configure x-axis to show model names at the center of the grouped bars
    xaxis=dict(
        tickmode='array',
        tickvals=[model_indices[m] for m in overalles_df['model_name']],
        ticktext=overalles_df['model_name'].tolist(),
        showline=True, linecolor='lightgray',
        automargin=True,
        title='Model', # Add title for x-axis
        range=[-0.5, len(overalles_df['model_name']) - 0.5] # Adjust x-axis range
    ),
    # Configure primary y-axis
    yaxis=dict(
        title='Chrf++',
        side='left',
        showline=True, linecolor='lightgray',
        gridcolor='lightgray',
        automargin=True
    ),
    # Configure secondary y-axis
    yaxis2=dict(
        title='Prop. of valid spanish translations',
        overlaying='y', # Overlay on primary y-axis
        side='right',
        showline=True, linecolor='lightgray',
        gridcolor='lightgray', # Grid should match primary y-axis
        automargin=True
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    plot_bgcolor='white',
    autosize=True,
    width=1000,
    height=600,
    bargap=0.05, # Gap between the two bars within a group
    bargroupgap=0.1 # Gap between different groups
)

fig.show()


## `RTTSacreBLEU` metric

In [ ]:
overalles_df['rttsacrebleu_round'] = round(overalles_df['rtt_sacrebleu'],2)

In [ ]:
dataes_df = overalles_df.sort_values(by='rtt_sacrebleu', ascending=False)
fig = px.bar(dataes_df, x='rtt_sacrebleu', y='model_name', template='none', text='rttsacrebleu_round',
             color='model_name', orientation='h', color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(autosize=True, xaxis_title='RTTSacreBLEU', yaxis_title='',
                  showlegend=False, yaxis=dict(tickfont=dict(size=13)), width=1200,
                  margin=dict(l=235))
fig.show()

## Analysis `RTTchrf++` metric

In [ ]:
dataes_df['rttchrf++_round'] = round(dataes_df['rtt_chrf++'],2)

In [ ]:
dataes_df = dataes_df.sort_values(by='rtt_chrf++', ascending=False)
fig = px.bar(dataes_df, x='rtt_chrf++', y='model_name', template='none', text='rttchrf++_round',
             color='model_name', orientation='h', color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(autosize=True, xaxis_title='RTTchrf++', yaxis_title='',
                  showlegend=False, yaxis=dict(tickfont=dict(size=13)), width=1200,
                  margin=dict(l=235))
fig.show()

## RTTchrf++ per Domain

In [ ]:
evaluation_domains_df['model'] = evaluation_domains_df['model'].apply(lambda x: x.split('/')[-1])
evaluation_domains_df = evaluation_domains_df.loc[~evaluation_domains_df['model'].isin(models_to_exclude)]

In [ ]:
# Identify the best model for each domain based on 'rtt_chrf++'
best_model_per_domain = evaluation_domains_df.loc[evaluation_domains_df.groupby('domain')['rtt_chrf++'].idxmax()]

# Get all model names from the best_model_per_domain_df
all_best_models = best_model_per_domain['model'].tolist()

# Count the occurrences of each model
model_counts = pd.Series(all_best_models).value_counts()

# Get a list of all unique model names from evaluation_domains_df
all_unique_models = evaluation_domains_df['model'].unique().tolist()

# Reindex the model_counts Series to include all models, filling NaN with 0
model_counts_all = model_counts.reindex(all_unique_models, fill_value=0)

# Convert the counts to a DataFrame, where model names are columns
best_model_counts_df = pd.DataFrame(model_counts_all).T

# Rename the index for clarity
best_model_counts_df.index = ['Domains Win']

In [ ]:
heatmap_data = evaluation_domains_df.pivot_table(index='domain', columns='model', values='rtt_chrf++')

# Calculate the macro average across domains for each model
macro_avg_chrf = evaluation_domains_df.groupby('model')['rtt_chrf++'].mean()

# Convert the series to a DataFrame row
macro_avg_df = pd.DataFrame(macro_avg_chrf).T
macro_avg_df.index = ['Macro Average']

# Create an empty row for separation
empty_row_1 = pd.DataFrame(np.nan, index=[''], columns=heatmap_data.columns)
empty_row_2 = pd.DataFrame(np.nan, index=[' '], columns=heatmap_data.columns)

# Concatenate original data, empty row, macro average row, another empty row, and best model counts
heatmap_data = pd.concat([heatmap_data, empty_row_1, macro_avg_df, empty_row_2, best_model_counts_df])

fig = px.imshow(heatmap_data,
                text_auto=True, # Show values in cells
                aspect="auto", # Adjust aspect ratio automatically
                color_continuous_scale=cl.diverging.RdYlGn, # Change color scale from red (low) to green (high)
                title='',
                labels={'x': 'Model', 'y': 'Domain', 'color': 'RTTchrf++'}) # Adjust y-axis label to 'Domain' after adding macro average

fig.update_layout(
    autosize=True,
    width=1200,
    height=800,
    xaxis_title='',
    yaxis_title='',
    plot_bgcolor='white',
    xaxis=dict(tickangle=-35, side='top'), # Place x-axis labels (models) on top
    coloraxis_colorbar=dict(
        title='Average chrf++ (higher is better)', # Color bar title
        titleside='right',
        tickprefix=' '
    )
)

fig.show()

In [ ]:
evaluation_domains_df = evaluation_domains_df.loc[~evaluation_domains_df['model'].isin(['gpt-4o-mini', 'grok-4-fast-non-reasoning'])]
# Identify the best model for each domain based on 'rtt_chrf++'
best_model_per_domain = evaluation_domains_df.loc[evaluation_domains_df.groupby('domain')['rtt_chrf++'].idxmax()]

# Get all model names from the best_model_per_domain_df
all_best_models = best_model_per_domain['model'].tolist()

# Count the occurrences of each model
model_counts = pd.Series(all_best_models).value_counts()

# Get a list of all unique model names from evaluation_domains_df
all_unique_models = evaluation_domains_df['model'].unique().tolist()

# Reindex the model_counts Series to include all models, filling NaN with 0
model_counts_all = model_counts.reindex(all_unique_models, fill_value=0)

# Convert the counts to a DataFrame, where model names are columns
best_model_counts_df = pd.DataFrame(model_counts_all).T

# Rename the index for clarity
best_model_counts_df.index = ['Domains Win']

In [ ]:
heatmap_data = evaluation_domains_df.pivot_table(index='domain', columns='model', values='rtt_chrf++')

# Calculate the macro average across domains for each model
macro_avg_chrf = evaluation_domains_df.groupby('model')['rtt_chrf++'].mean()

# Convert the series to a DataFrame row
macro_avg_df = pd.DataFrame(macro_avg_chrf).T
macro_avg_df.index = ['Macro Average']

# Create an empty row for separation
empty_row_1 = pd.DataFrame(np.nan, index=[''], columns=heatmap_data.columns)
empty_row_2 = pd.DataFrame(np.nan, index=[' '], columns=heatmap_data.columns)

# Concatenate original data, empty row, macro average row, another empty row, and best model counts
heatmap_data = pd.concat([heatmap_data, empty_row_1, macro_avg_df, empty_row_2, best_model_counts_df])

fig = px.imshow(heatmap_data,
                text_auto=True, # Show values in cells
                aspect="auto", # Adjust aspect ratio automatically
                color_continuous_scale=cl.diverging.RdYlGn, # Change color scale from red (low) to green (high)
                title='Open-weight models',
                labels={'x': 'Model', 'y': 'Domain', 'color': 'RTTchrf++'}) # Adjust y-axis label to 'Domain' after adding macro average

fig.update_layout(
    autosize=True,
    width=1200,
    height=800,
    xaxis_title='',
    yaxis_title='',
    plot_bgcolor='white',
    xaxis=dict(tickangle=-35, side='top'), # Place x-axis labels (models) on top
    coloraxis_colorbar=dict(
        title='Average chrf++ (higher is better)', # Color bar title
        titleside='right',
        tickprefix=' '
    )
)

fig.show()